# NIFTY100 Financial Analytics

## Sprint 2 – Day 9

# Advanced Analytics Engine

This notebook combines financial performance,
growth analytics and risk analytics
to generate final company rankings.

### Objectives

- Merge scorecards
- Rank companies
- Calculate Value at Risk
- Classify Investment Type
- Export Final Analytics Dataset


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
ROOT = Path("../")

OUTPUT = ROOT / "outputs"

performance = pd.read_csv(
    OUTPUT / "company_scorecard.csv"
)

growth = pd.read_csv(
    OUTPUT / "growth_scorecard.csv"
)

risk = pd.read_csv(
    OUTPUT / "risk_scorecard.csv"
)

print(performance.shape)
print(growth.shape)
print(risk.shape)

(1191, 6)
(100, 5)
(100, 6)


## Overall Company Ranking

Combine Performance, Growth and Risk scorecards into one ranking.

In [4]:
# Keep the latest available record for each company

performance_latest = (
    performance
    .sort_values("year")
    .groupby("company_id")
    .tail(1)
)

performance_latest = performance_latest[
    [
        "company_id",
        "Performance_Score"
    ]
]

performance_latest.head()

,company_id,Performance_Score
21,INDIGO,5319.928
2,COALINDIA,14427.668
762,DMART,19.524
15,BAJFINANCE,6073.536
1175,KOTAKBANK,-3981.396


In [5]:
# Merge all three scorecards

ranking = performance_latest.merge(
    growth[
        [
            "company_id",
            "growth_score"
        ]
    ],
    on="company_id",
    how="outer"
)

ranking = ranking.merge(
    risk[
        [
            "company_id",
            "risk_score"
        ]
    ],
    on="company_id",
    how="outer"
)

ranking.fillna(0, inplace=True)

ranking.head()

,company_id,Performance_Score,growth_score,risk_score
0,ABB,189.988,4.281683,577.580626
1,ADANIENSOL,15.784,-7.072847,2774.060155
2,ADANIENT,15.412,35.438953,12061.888493
3,ADANIGREEN,30.924,27.982922,1489.101946
4,ADANIPORTS,34.940,0.000000,3145.806394


In [6]:
# Calculate Overall Score

ranking["overall_score"] = (

    ranking["Performance_Score"] * 0.40

    +

    ranking["growth_score"] * 0.35

    -

    ranking["risk_score"] * 0.25

)

ranking = ranking.sort_values(
    by="overall_score",
    ascending=False
)

ranking.reset_index(
    drop=True,
    inplace=True
)

ranking["rank"] = ranking.index + 1

ranking.head(20)

,company_id,Performance_Score,growth_score,risk_score,overall_score,rank
0,COALINDIA,14427.668,-3.792529,10803.945328,3068.753483,1
1,BAJFINANCE,6073.536,7.913249,7672.010722,514.181356,2
2,HINDUNILVR,4418.928,2.342267,5183.738846,472.456282,3
3,BEL,1906.620,8.160364,2098.775705,240.810201,4
4,HEROMOTOCO,1635.056,6.623681,1747.970814,219.347985,5
5,CIPLA,1908.820,9.894390,2345.399098,180.641262,6
6,HAL,1570.432,5.636570,2261.626427,64.738993,7
7,BAJAJHLDNG,228.932,1.385717,218.838203,37.348250,8
8,CHOLAFIN,1383.684,12.022313,2478.949967,-62.056082,9
9,ABB,189.988,4.281683,577.580626,-66.901367,10


In [7]:
ranking.to_csv(
    OUTPUT / "company_ranking.csv",
    index=False
)

print("Company Ranking Saved Successfully!")

Company Ranking Saved Successfully!


## Value at Risk (VaR)

Value at Risk (VaR) estimates the maximum expected loss over a given period at a chosen confidence level.

For this project, VaR is estimated using the 5th percentile of Year-over-Year Sales Growth for each company.

In [8]:
import sqlite3

DATABASE = ROOT / "data" / "db" / "nifty100.db"

conn = sqlite3.connect(DATABASE)

profit = pd.read_sql(
    "SELECT company_id, year, sales FROM profitandloss",
    conn
)

conn.close()

profit.head()

,company_id,year,sales
0,ABB,2012.0,1653.0
1,ABB,2014.0,2276.0
2,ABB,2015.0,2289.0
3,ABB,2016.0,2614.0
4,ABB,2017.0,2903.0


In [9]:
profit = profit.sort_values(
    ["company_id", "year"]
)

profit["sales_growth"] = (
    profit
    .groupby("company_id")["sales"]
    .pct_change()
)

profit.head(15)

,company_id,year,sales,sales_growth
0,ABB,2012.0,1653.0,NaN
1,ABB,2014.0,2276.0,0.376891
2,ABB,2015.0,2289.0,0.005712
3,ABB,2016.0,2614.0,0.141983
4,ABB,2017.0,2903.0,0.110559
5,ABB,2018.0,3298.0,0.136066
6,ABB,2019.0,3679.0,0.115525
7,ABB,2020.0,4093.0,0.112531
8,ABB,2021.0,4310.0,0.053017
9,ABB,2022.0,4913.0,0.139907


In [10]:
var_analysis = (
    profit
    .groupby("company_id")["sales_growth"]
    .quantile(0.05)
    .reset_index()
)

var_analysis.rename(
    columns={
        "sales_growth": "value_at_risk"
    },
    inplace=True
)

var_analysis["value_at_risk"] = (
    var_analysis["value_at_risk"] * 100
).round(2)

var_analysis = var_analysis.sort_values(
    by="value_at_risk"
)

var_analysis.head(20)

,company_id,value_at_risk
45,ICICIPRULI,-43.90
50,IRCTC,-37.72
2,ADANIENT,-34.68
39,HDFCLIFE,-28.42
13,BAJAJHLDNG,-28.22
59,LODHA,-27.45
90,TRENT,-25.87
18,BHEL,-24.72
75,RELIANCE,-24.25
49,IOC,-23.80


In [11]:
var_analysis.to_csv(
    OUTPUT / "value_at_risk.csv",
    index=False
)

print("Value at Risk Analysis Saved Successfully!")

Value at Risk Analysis Saved Successfully!


## Investment Category

Companies are classified into different investment categories based on
their Growth Score and Risk Score.

Categories:

- Growth
- Value
- Stable
- High Risk

In [12]:
investment = ranking.merge(
    var_analysis,
    on="company_id",
    how="left"
)

investment.head()

,company_id,Performance_Score,growth_score,risk_score,overall_score,rank,value_at_risk
0,COALINDIA,14427.668,-3.792529,10803.945328,3068.753483,1,-4.77
1,BAJFINANCE,6073.536,7.913249,7672.010722,514.181356,2,6.61
2,HINDUNILVR,4418.928,2.342267,5183.738846,472.456282,3,0.96
3,BEL,1906.620,8.160364,2098.775705,240.810201,4,3.80
4,HEROMOTOCO,1635.056,6.623681,1747.970814,219.347985,5,-9.22


In [13]:
def classify_company(row):

    # High Growth + Low Risk
    if row["growth_score"] >= 20 and row["risk_score"] < 2000:
        return "Growth"

    # High Performance + Moderate Risk
    elif row["Performance_Score"] >= 100 and row["risk_score"] < 5000:
        return "Value"

    # Very High Risk
    elif row["risk_score"] >= 5000:
        return "High Risk"

    # Everything else
    else:
        return "Stable"

In [14]:
investment["investment_category"] = investment.apply(
    classify_company,
    axis=1
)

investment.head(20)

,company_id,Performance_Score,growth_score,risk_score,overall_score,rank,value_at_risk,investment_category
0,COALINDIA,14427.668,-3.792529,10803.945328,3068.753483,1,-4.77,High Risk
1,BAJFINANCE,6073.536,7.913249,7672.010722,514.181356,2,6.61,High Risk
2,HINDUNILVR,4418.928,2.342267,5183.738846,472.456282,3,0.96,High Risk
3,BEL,1906.620,8.160364,2098.775705,240.810201,4,3.80,Value
4,HEROMOTOCO,1635.056,6.623681,1747.970814,219.347985,5,-9.22,Value
5,CIPLA,1908.820,9.894390,2345.399098,180.641262,6,4.47,Value
6,HAL,1570.432,5.636570,2261.626427,64.738993,7,2.89,Value
7,BAJAJHLDNG,228.932,1.385717,218.838203,37.348250,8,-28.22,Value
8,CHOLAFIN,1383.684,12.022313,2478.949967,-62.056082,9,8.15,Value
9,ABB,189.988,4.281683,577.580626,-66.901367,10,2.30,Value


In [15]:
investment["investment_category"].value_counts()

investment_category
High Risk    51
Stable       35
Value        10
Growth        4
Name: count, dtype: int64

In [16]:
investment.to_csv(
    OUTPUT / "investment_category.csv",
    index=False
)

print("Investment Category Saved Successfully!")

Investment Category Saved Successfully!


## Final Analytics Dataset

This dataset combines:

- Company Ranking
- Performance Score
- Growth Score
- Risk Score
- Value at Risk
- Investment Category

It serves as the master dataset for downstream analytics and recommendation systems.

In [17]:
final_analytics = investment[
    [
        "rank",
        "company_id",
        "Performance_Score",
        "growth_score",
        "risk_score",
        "overall_score",
        "value_at_risk",
        "investment_category"
    ]
].copy()

final_analytics.head(10)

,rank,company_id,Performance_Score,growth_score,risk_score,overall_score,value_at_risk,investment_category
0,1,COALINDIA,14427.668,-3.792529,10803.945328,3068.753483,-4.77,High Risk
1,2,BAJFINANCE,6073.536,7.913249,7672.010722,514.181356,6.61,High Risk
2,3,HINDUNILVR,4418.928,2.342267,5183.738846,472.456282,0.96,High Risk
3,4,BEL,1906.620,8.160364,2098.775705,240.810201,3.80,Value
4,5,HEROMOTOCO,1635.056,6.623681,1747.970814,219.347985,-9.22,Value
5,6,CIPLA,1908.820,9.894390,2345.399098,180.641262,4.47,Value
6,7,HAL,1570.432,5.636570,2261.626427,64.738993,2.89,Value
7,8,BAJAJHLDNG,228.932,1.385717,218.838203,37.348250,-28.22,Value
8,9,CHOLAFIN,1383.684,12.022313,2478.949967,-62.056082,8.15,Value
9,10,ABB,189.988,4.281683,577.580626,-66.901367,2.30,Value


In [18]:
print(final_analytics.shape)

final_analytics.info()

(100, 8)
<class 'pandas.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 8 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   rank                 100 non-null    int64  
 1   company_id           100 non-null    str    
 2   Performance_Score    100 non-null    float64
 3   growth_score         100 non-null    float64
 4   risk_score           100 non-null    float64
 5   overall_score        100 non-null    float64
 6   value_at_risk        100 non-null    float64
 7   investment_category  100 non-null    str    
dtypes: float64(5), int64(1), str(2)
memory usage: 6.4 KB


In [19]:
final_analytics = final_analytics.sort_values(
    by="rank"
)

final_analytics.reset_index(
    drop=True,
    inplace=True
)

final_analytics.head(20)

,rank,company_id,Performance_Score,growth_score,risk_score,overall_score,value_at_risk,investment_category
0,1,COALINDIA,14427.668,-3.792529,10803.945328,3068.753483,-4.77,High Risk
1,2,BAJFINANCE,6073.536,7.913249,7672.010722,514.181356,6.61,High Risk
2,3,HINDUNILVR,4418.928,2.342267,5183.738846,472.456282,0.96,High Risk
3,4,BEL,1906.620,8.160364,2098.775705,240.810201,3.80,Value
4,5,HEROMOTOCO,1635.056,6.623681,1747.970814,219.347985,-9.22,Value
5,6,CIPLA,1908.820,9.894390,2345.399098,180.641262,4.47,Value
6,7,HAL,1570.432,5.636570,2261.626427,64.738993,2.89,Value
7,8,BAJAJHLDNG,228.932,1.385717,218.838203,37.348250,-28.22,Value
8,9,CHOLAFIN,1383.684,12.022313,2478.949967,-62.056082,8.15,Value
9,10,ABB,189.988,4.281683,577.580626,-66.901367,2.30,Value


In [20]:
final_analytics.to_csv(
    OUTPUT / "final_analytics.csv",
    index=False
)

print("Final Analytics Dataset Saved Successfully!")

Final Analytics Dataset Saved Successfully!
